In [1]:
import polars as pl

In [2]:
def perform_transformation(dataframe):
    """Performs a transformation to get the cartesian components and magnitude of the momentum and
    calculates the effective mass of the systems of dimuons from data which is in cylindrical coordinates.

    Args:
        dataframe (polars.DataFrame): a polars DataFrame of the format:
            transverse momentum 1, azimuth angle 1, pseudorapidity 1,
            transverse momentum 2, azimuth angle 2, pseudorapidity 2, effective mass

            with the column names:
            p_T1, phi_1, eta_1, p_T2, phi_2, eta_2, effective_mass

    Returns:
        polars.DataFrame: A polars DataFrame with columns specified below:
            p1_x, p1_z, p1_y, |p_1|, p2_x, p2_z, p2_y, |p_2|, calculated_effective_mass, effective_mass
    """
    p_t_1 = dataframe["p_T1"]
    p_t_2 = dataframe["p_T2"]
    phi_1 = dataframe["phi_1"]
    phi_2 = dataframe["phi_2"]
    eta_1 = dataframe["eta_1"]
    eta_2 = dataframe["eta_2"]

    p1_x = (p_t_1 * phi_1.cos()).alias("p1_x")
    p1_y = (p_t_1 * phi_1.sin()).alias("p1_y")
    p1_z = (p_t_1 * eta_1.sinh()).alias("p1_z")
    p1 = (p_t_1 * eta_1.cosh()).alias("|p1|")

    p2_x = (p_t_2 * phi_2.cos()).alias("p2_x")
    p2_y = (p_t_2 * phi_2.sin()).alias("p2_y")
    p2_z = (p_t_2 * eta_2.sinh()).alias("p2_z")
    p2 = (p_t_2 * eta_2.cosh()).alias("|p2|")

    # use approximation m_mu is negligible compared to momentum, E ~= p for high energy muon
    calculated_effective_mass = (
        ((p1 + p2)**2 - (p1_x + p2_x)**2 - (p1_y + p2_y)**2 - (p1_z + p2_z)**2 )
        .sqrt().alias("calculated_effective_mass")
    )

    return pl.DataFrame(
        [p1_x, p1_y, p1_z, p1, p2_x, p2_y, p2_z, p2, calculated_effective_mass,
         dataframe.get_column("effective_mass"),]
    )


In [3]:
huge_data = pl.read_csv("data/real-dimuon-29M.data", separator="\t", low_memory=True,
                            new_columns=["p_T1", "phi_1", "eta_1", "p_T2", "phi_2", "eta_2",
                                         "effective_mass"])

In [4]:
huge_data_transformed = perform_transformation(huge_data)